In [9]:
from faker import Faker
from faker.providers import DynamicProvider
import pandas as pd
from random import randint
from pathlib import Path

In [10]:
fake = Faker(['fr_FR'])
nbr_of_samples = 1_000_000
nbr_max_hobbies = 5
nbr_max_traits = 5

# Dossier de sauvegarde
dossier = Path("Fake_profiles")

# Path to CSV files
cities_csv_path = 'csv/communes-france-2025.csv'
hobbys_csv_path = 'ChatGPT/liste_hobbies.csv'
traits_csv_path = 'ChatGPT/liste_traits_caractere.csv'
jobs_csv_path = 'ChatGPT/liste_metiers.csv'

In [11]:
# Load city names from CSV
df_cities = pd.read_csv(cities_csv_path, sep=',', usecols=['nom_standard'])
print(df_cities.shape)
cities_list = df_cities['nom_standard'].tolist()

df_hobbys = pd.read_csv(hobbys_csv_path, sep=',', usecols=['Hobby'])
print(df_hobbys.shape)
hobbys_list = df_hobbys['Hobby'].tolist()

# Load character traits from CSV
df_traits = pd.read_csv(traits_csv_path, sep=',', usecols=['Trait'])
print(df_traits.shape)
traits_list = df_traits['Trait'].tolist()

# Load job titles from CSV
df_jobs = pd.read_csv(jobs_csv_path, sep=',', usecols=['Metier'])
print(df_jobs.shape)
jobs_list = df_jobs['Metier'].tolist()

(34935, 1)
(210, 1)
(84, 1)
(191, 1)


In [12]:
# Providers

# City Provider
city_provider = DynamicProvider(
    provider_name="city",
    elements=cities_list
)
fake.add_provider(city_provider)

# Hobby Provider
hobby_provider = DynamicProvider(
    provider_name="hobby",
    elements=hobbys_list
)
fake.add_provider(hobby_provider)

# Trait Provider
trait_provider = DynamicProvider(
    provider_name="trait",
    elements=traits_list
)
fake.add_provider(trait_provider)

# Job Provider
job_provider = DynamicProvider(
    provider_name="job",
    elements=jobs_list
)
fake.add_provider(job_provider)

In [13]:
# Generate a sample fake profile
nom_prenom = fake.name().split(" ")
print("Id :", fake.uuid4())
print("Prénom :", fake.first_name())
print("Nom :", fake.last_name())
print("Sexe :", fake.random_element(elements=["M", "F"]))
print("Âge :", fake.random_int(min=18, max=80))
print("Ville :", fake.city())
print("Hobby :", fake.hobby())
print("Trait :", fake.trait())
print("Job :", fake.job())

Id : 331da705-24bd-4a03-adcf-d0a88daef680
Prénom : Joséphine
Nom : Riou
Sexe : F
Âge : 46
Ville : Villars-et-Villenotte
Hobby : Maroquinerie
Trait : Bienveillant
Job : Professeur des écoles


In [14]:
df_fake_profile = pd.DataFrame({
    "Id": [fake.uuid4() for _ in range(nbr_of_samples)],
    "Prénom": [fake.first_name() for _ in range(nbr_of_samples)],
    "Nom": [fake.last_name() for _ in range(nbr_of_samples)],
    "Sexe": [fake.random_element(elements=["M", "F"]) for _ in range(nbr_of_samples)],
    "Âge": [fake.random_int(min=18, max=80) for _ in range(nbr_of_samples)],
    "Ville": [fake.city() for _ in range(nbr_of_samples)],
    "Hobby": [[fake.hobby() for _ in range(randint(1, nbr_max_hobbies))] for _ in range(nbr_of_samples)],
    "Trait": [[fake.trait() for _ in range(randint(1, nbr_max_traits))] for _ in range(nbr_of_samples)],
    "Job": [fake.job() for _ in range(nbr_of_samples)]
})
print(df_fake_profile.columns)
print(df_fake_profile.shape)
df_fake_profile.head()
# Temps (100_000): 
# Temps (1_000_000): 2min 30s

Index(['Id', 'Prénom', 'Nom', 'Sexe', 'Âge', 'Ville', 'Hobby', 'Trait', 'Job'], dtype='object')
(1000000, 9)


,Id,Prénom,Nom,Sexe,Âge,Ville,Hobby,Trait,Job
0,c9222594-239f-4301-82a6-694de92bfaf9,Eugène,Delannoy,F,39,Rapilly,[Sculpture],[Négatif],Traiteur
1,883dccc0-ebe1-41d4-8237-60a7c3833774,Pénélope,Deschamps,M,19,Gauriaguet,"[Poésie, Ski de fond]",[Susceptible],Électricien
2,c042f38c-4763-452e-afb3-f93338a84d43,Maurice,Ollivier,F,61,Gignac,[Fitness],"[Conservateur, Têtu, Compétitif]",Développeur web
3,f6d327b1-fed4-4a9c-97ff-b6ad7c110b5a,Claude,Besnard,F,61,Vaunac,"[Fitness, Origami]","[Impatient, Contemplatif, Enthousiaste]",Comédien
4,e5b74ff1-e0b9-4d15-8a04-8bcd8156190c,Julien,Verdier,M,35,Tavant,"[Streaming, Apnée, Œnologie]","[Extraverti, Discipliné, Indiscipliné, Tolérant]",Commercial


In [15]:
if not dossier.exists():
    dossier.mkdir(parents=True)
    print("Dossier créé")
else:
    print("Le dossier existe déjà")

Dossier créé


In [16]:
# Save to CSV or Parquet
# df_fake_profile.to_csv(f"{dossier}/fake_profiles_{nbr_of_samples:_}.csv", index=False)

# Plus opti pour la taille et la vitesse de lecture/écriture et pour la sauvegare des listes
# df_fake_profile.to_parquet(f"{dossier}/fake_profiles_{nbr_of_samples:_}.parquet", index=False)
# Temps (1_000_000): 

# JSON
df_fake_profile.to_json(f"{dossier}/fake_profiles_{nbr_of_samples:_}.json", orient="records")
# Temps (1_000_000): 4.3s